In [1]:
import csv
import re
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

In [2]:
# ===== CONFIG =====
URL_FILE      = "cryptomasun.txt"   # input: one URL per line
OUTPUT_CSV    = None                       # None => auto: "<URL_FILE stem>_decoded.csv"
TEST_ONLY     = False                      # if True, process only first URL (for quick sanity checks)
# ==================

VIDEO_ID_RE = re.compile(r"/video/(\d+)")
USERNAME_RE = re.compile(r"/@([^/]+)")

def extract_video_id(url: str) -> str | None:
    """
    Works for URLs containing '/video/<id>'.
    """
    m = VIDEO_ID_RE.search(url)
    return m.group(1) if m else None

def extract_username(url: str) -> str | None:
    """
    Extracts '@username' segment if present (e.g., /@someuser/video/...)
    """
    try:
        path = urlparse(url).path
    except Exception:
        return None
    m = USERNAME_RE.search(path)
    return m.group(1) if m else None

def decode_tiktok_upload_time(video_id: str) -> datetime:
    """
    upload_time (Unix seconds) = video_id >> 32
    """
    secs = int(video_id) >> 32
    return datetime.fromtimestamp(secs, tz=timezone.utc)

def main():
    in_path = Path(URL_FILE)
    if not in_path.exists():
        print(f"⚠️ Input file not found: {in_path}")
        return

    out_path = Path(OUTPUT_CSV) if OUTPUT_CSV else in_path.with_name(in_path.stem + "_decoded.csv")

    with in_path.open("r", encoding="utf-8") as f:
        urls = [u.strip() for u in f if u.strip()]

    if not urls:
        print("⚠️ No URLs found.")
        return

    if TEST_ONLY:
        urls = urls[:1]

    rows = []
    for idx, url in enumerate(urls, start=1):
        vid = extract_video_id(url)
        if not vid:
            # If the URL is a short/redirect (vm.tiktok.com, etc.), we can't decode without resolving.
            print(f"[{idx}/{len(urls)}] Skipped (no /video/<id>): {url}")
            rows.append({"url": url, "user": extract_username(url) or "", "timestamp": ""})
            continue

        try:
            upload_dt = decode_tiktok_upload_time(vid)
            iso_utc = upload_dt.strftime("%Y-%m-%dT%H:%M:%SZ")  # ISO 8601 UTC with 'Z'
        except Exception as e:
            print(f"[{idx}/{len(urls)}] Error decoding ID {vid}: {e}")
            iso_utc = ""

        user = extract_username(url) or ""
        rows.append({"url": url, "user": user, "timestamp": iso_utc})
        print(f"[{idx}/{len(urls)}] ✓ {user or '(no user)'} | {iso_utc}")

    # write CSV
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["url", "user", "timestamp"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"\n✅ Done. Wrote {len(rows)} rows to: {out_path}")

if __name__ == "__main__":
    main()

[1/538] ✓ cryptomasun | 2025-11-11T15:13:26Z
[2/538] ✓ cryptomasun | 2025-11-09T13:29:27Z
[3/538] ✓ cryptomasun | 2025-11-07T18:03:24Z
[4/538] ✓ cryptomasun | 2025-11-06T21:29:45Z
[5/538] ✓ cryptomasun | 2025-11-06T17:29:58Z
[6/538] ✓ cryptomasun | 2025-11-06T17:29:08Z
[7/538] ✓ cryptomasun | 2025-11-03T19:23:57Z
[8/538] ✓ cryptomasun | 2025-10-28T21:45:25Z
[9/538] ✓ cryptomasun | 2025-10-28T14:31:35Z
[10/538] ✓ cryptomasun | 2025-10-24T19:06:52Z
[11/538] ✓ cryptomasun | 2025-10-24T16:14:29Z
[12/538] ✓ cryptomasun | 2025-10-24T16:08:05Z
[13/538] ✓ cryptomasun | 2025-10-24T16:02:12Z
[14/538] ✓ cryptomasun | 2025-10-24T14:57:57Z
[15/538] ✓ cryptomasun | 2025-10-22T13:23:35Z
[16/538] ✓ cryptomasun | 2025-10-20T14:25:47Z
[17/538] ✓ cryptomasun | 2025-10-17T15:59:07Z
[18/538] ✓ cryptomasun | 2025-10-16T16:10:12Z
[19/538] ✓ cryptomasun | 2025-10-15T19:12:08Z
[20/538] ✓ cryptomasun | 2025-10-08T22:33:16Z
[21/538] ✓ cryptomasun | 2025-10-05T18:15:20Z
[22/538] ✓ cryptomasun | 2025-10-04T20:58:5

In [2]:
# ===== CONFIG =====
INPUT_FOLDER = "input_txts"        # folder containing .txt files with URLs
OUTPUT_CSV   = "combined_decoded.csv"
TEST_ONLY    = False
# ==================

VIDEO_ID_RE = re.compile(r"/video/(\d+)")
USERNAME_RE = re.compile(r"/@([^/]+)")

def extract_video_id(url: str) -> str | None:
    m = VIDEO_ID_RE.search(url)
    return m.group(1) if m else None

def extract_username(url: str) -> str | None:
    try:
        path = urlparse(url).path
    except:
        return None
    m = USERNAME_RE.search(path)
    return m.group(1) if m else None

def decode_tiktok_upload_time(video_id: str) -> datetime:
    secs = int(video_id) >> 32
    return datetime.fromtimestamp(secs, tz=timezone.utc)

def process_file(txt_path: Path):
    """Returns a list of rows from a single txt file."""
    with txt_path.open("r", encoding="utf-8") as f:
        urls = [u.strip() for u in f if u.strip()]

    if TEST_ONLY:
        urls = urls[:1]

    rows = []
    for url in urls:
        vid = extract_video_id(url)
        if vid:
            # decode
            try:
                dt = decode_tiktok_upload_time(vid)
                iso_utc = dt.strftime("%Y-%m-%dT%H:%M:%SZ")
            except:
                iso_utc = ""
        else:
            iso_utc = ""

        rows.append({
            "source_file": txt_path.name,
            "url": url,
            "user": extract_username(url) or "",
            "timestamp": iso_utc
        })

    return rows

def main():
    folder = Path(INPUT_FOLDER)
    if not folder.exists():
        print(f"⚠️ Folder not found: {INPUT_FOLDER}")
        return

    # find all txt files
    txt_files = sorted(folder.glob("*.txt"))
    if not txt_files:
        print("⚠️ No .txt files found in folder.")
        return

    all_rows = []

    print(f"Found {len(txt_files)} txt files.\n")

    for txt in txt_files:
        print(f"📄 Processing {txt.name} ...")
        rows = process_file(txt)
        all_rows.extend(rows)
        print(f"  → {len(rows)} rows")

    # Write final CSV
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["source_file", "url", "user", "timestamp"])
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n✅ Finished. Total {len(all_rows)} rows written to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

Found 11 txt files.

📄 Processing coin_guide.txt ...
  → 2543 rows
📄 Processing coinbureau.txt ...
  → 417 rows
📄 Processing crypto_jiggy.txt ...
  → 67 rows
📄 Processing forrestunfiltered.txt ...
  → 336 rows
📄 Processing girlgone_crypto.txt ...
  → 912 rows
📄 Processing im.cryptochino.txt ...
  → 465 rows
📄 Processing imcameronscrubs.txt ...
  → 410 rows
📄 Processing layahheilpernofficial.txt ...
  → 161 rows
📄 Processing layahtrades.txt ...
  → 186 rows
📄 Processing theblockchainboy.txt ...
  → 2108 rows
📄 Processing titovlogs77.txt ...
  → 1651 rows

✅ Finished. Total 9256 rows written to combined_decoded.csv
